# NB_03 — Silver Transformation

**Project:** Nexa Commerce  
**Lakehouse:** Nexa_Lakehouse  
**Layer:** Silver  

Transform the Bronze source tables into clean and standardized Silver tables.

The transformations are based on the issues identified during Bronze profiling.

Bronze remains unchanged.

### 1 — ChannelAdvisor Channel Names

Standardize the different ChannelAdvisor marketplace names into consistent channel names.

In [12]:
%%sql
CREATE OR REPLACE TABLE silver_channeladvisor_orders
USING DELTA
AS 
SELECT DISTINCT
source_order_id,
channel_order_id,
CASE 
    WHEN channel_name IN ('HOME DEPOT','HomeDepot', 'home depot' ) THEN 'Home Depot'
    WHEN channel_name IN ('target', 'TARGET','Target.com' ) THEN 'Target'
    WHEN channel_name = 'AMAZON'  THEN 'Amazon'
    ELSE channel_name
END AS channel_name,
order_datetime,
ship_date,
currency,
order_status,
subtotal,
tax_amount,
shipping_amount
discount_amount,
order_total

FROM bronze_channeladvisor_orders

StatementMeta(, c8131984-1207-4336-acb2-762ff966bbca, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [13]:
SELECT "Bronze_ChannelAdvisor" AS channel_name,
COUNT(source_order_id)
FROM bronze_channeladvisor_orders

UNION ALL

SELECT "Silver_ChannelAdvisor" AS channel_name,
COUNT(source_order_id)
FROM silver_channeladvisor_orders


StatementMeta(, c8131984-1207-4336-acb2-762ff966bbca, 14, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

### 2 — Currency Normalization
Identify the Shopify records using CAD and convert them to USD in the Silver layer.

In [3]:
CREATE OR REPLACE TABLE silver_shopify_order_lines
USING DELTA
AS
SELECT
    ol.source_order_id,
    ol.line_number,
    ol.sku,
    ol.quantity,
    so.currency,
    ce.exchange_rate,

    ol.unit_price,
    ol.line_discount,
    ol.line_tax,
    ol.line_total,

    ROUND(ol.unit_price * ce.exchange_rate, 2) AS unit_price_usd,
    ROUND(ol.line_discount * ce.exchange_rate, 2) AS line_discount_usd,
    ROUND(ol.line_tax * ce.exchange_rate, 2) AS line_tax_usd,
    ROUND(ol.line_total * ce.exchange_rate, 2) AS line_total_usd

FROM bronze_shopify_order_lines ol
INNER JOIN bronze_shopify_orders so
    ON ol.source_order_id = so.source_order_id

LEFT JOIN currency_exchange ce
    ON so.currency = ce.currency;



StatementMeta(, ee6c36e1-267b-478c-8732-9e39d8cca077, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
SELECT 
    source_order_id,
    sku,
    currency,
    exchange_rate,
    line_total,
    line_total_usd
FROM silver_shopify_order_lines
WHERE currency = 'CAD';

StatementMeta(, ee6c36e1-267b-478c-8732-9e39d8cca077, 6, Finished, Available, Finished, False)

<Spark SQL result set with 21 rows and 6 fields>

In [2]:
CREATE OR REPLACE TABLE currency_exchange
USING DELTA
AS 
SELECT 
"USD" AS currency,
"USD" AS target_currency,
1.00 AS exchange_rate

UNION ALL

SELECT 
"CAD" AS currency,
"USD" AS target_currency,
0.73 AS exchange_rate


StatementMeta(, c95c0953-9b03-4b50-9930-a3fce36a1c85, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
SELECT DISTINCT
currency,
exchange_rate
FROM silver_shopify_orders

StatementMeta(, dfaf9d5c-6c6c-439a-92fe-8066c8796bdf, 2, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

### 3 — Refund SKU Standardization

Replace missing SKU values with `N/A` while keeping existing SKU values unchanged.

In [4]:
CREATE OR REPLACE TABLE silver_refund_events
USING DELTA
AS
SELECT
refund_id,
source_channel,
source_order_id,
refund_date,
CASE 
WHEN sku IS NULL THEN "N/A"
ELSE sku
END sku,
CASE
WHEN quantity_refunded IS NULL THEN "N/A"
ELSE quantity_refunded
END quantity_refunded,
refund_amount,
currency,
refund_reason
FROM bronze_refund_events



StatementMeta(, dfaf9d5c-6c6c-439a-92fe-8066c8796bdf, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
SELECT *,
sku
FROM silver_refund_events
WHERE sku != "N/A"
LIMIT 3

StatementMeta(, dfaf9d5c-6c6c-439a-92fe-8066c8796bdf, 9, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 10 fields>

### 4 — Bronze to Silver Tables

Move the validated Bronze tables into the Silver layer. Apply the identified transformations where required and use pass-through `SELECT *` for tables that require no additional cleansing.

Validate the Silver layer by reconciling each Silver table with its corresponding Bronze table and confirming that the expected tables and row counts are present.

In [4]:
CREATE OR REPLACE TABLE silver_amazon_order_lines
USING DELTA
AS
SELECT *
FROM bronze_amazon_order_lines;

CREATE OR REPLACE TABLE silver_amazon_orders
USING DELTA
AS
SELECT *
FROM bronze_amazon_orders;

CREATE OR REPLACE TABLE silver_walmart_orders
USING DELTA
AS
SELECT *
FROM bronze_walmart_orders;

CREATE OR REPLACE TABLE silver_walmart_order_lines
USING DELTA
AS
SELECT *
FROM bronze_walmart_order_lines;

CREATE OR REPLACE TABLE silver_channeladvisor_order_lines
USING DELTA
AS
SELECT *
FROM bronze_channeladvisor_order_lines;

CREATE OR REPLACE TABLE silver_shopify_order_lines
USING DELTA
AS
SELECT *
FROM bronze_shopify_order_lines;

CREATE OR REPLACE TABLE silver_campaigns
USING DELTA
AS
SELECT *
FROM bronze_campaigns;

CREATE OR REPLACE TABLE silver_google_ads_campaign_daily
USING DELTA
AS
SELECT *
FROM bronze_google_ads_campaign_daily;

CREATE OR REPLACE TABLE silver_inventory_daily
USING DELTA
AS
SELECT *
FROM bronze_inventory_daily;

CREATE OR REPLACE TABLE silver_locations
USING DELTA
AS
SELECT *
FROM bronze_locations;

CREATE OR REPLACE TABLE silver_product_master
USING DELTA
AS
SELECT *
FROM bronze_product_master;

CREATE OR REPLACE TABLE silver_purchase_order_lines
USING DELTA
AS
SELECT *
FROM bronze_purchase_order_lines;

CREATE OR REPLACE TABLE silver_purchase_orders
USING DELTA
AS
SELECT *
FROM bronze_purchase_orders;


CREATE OR REPLACE TABLE silver_suppliers
USING DELTA
AS
SELECT *
FROM bronze_suppliers;

CREATE OR REPLACE TABLE silver_walmart_connect_weekly
USING DELTA
AS
SELECT *
FROM bronze_walmart_connect_weekly;


StatementMeta(, 22c5e87e-9b0c-4b83-9f36-b23015cec333, 38, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

...

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%pyspark
from pyspark.sql import Row

tables = spark.sql("SHOW TABLES").collect()

table_names = {row.tableName for row in tables}

results = []

for bronze in sorted(t for t in table_names if t.startswith("bronze_")):
    silver = bronze.replace("bronze_", "silver_", 1)

    bronze_count = spark.table(bronze).count()

    if silver in table_names:
        silver_count = spark.table(silver).count()
        status = "MATCH" if bronze_count == silver_count else "COUNT MISMATCH"
    else:
        silver_count = None
        status = "SILVER MISSING"

    results.append(
        Row(
            bronze_table=bronze,
            silver_table=silver,
            bronze_count=bronze_count,
            silver_count=silver_count,
            status=status
        )
    )

display(spark.createDataFrame(results))


StatementMeta(, 22c5e87e-9b0c-4b83-9f36-b23015cec333, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 999f34d1-05a7-423b-b4f0-3a006bbf3fd8)